# ⚡ LangChain Framework RAG with Groq + Gradio — Colab Notebook

This notebook converts the folder-wise VS Code task into a **Google Colab workflow**.

You will run the complete RAG pipeline:

```text
Data Ingestion → Data Transformation → Embeddings → FAISS Vector DB → Groq Answer → Gradio UI
```

This notebook is suitable for classroom demonstration and student practice.


## Step 1 — Install Required Packages

Run this cell first. It installs LangChain, Groq, FAISS, Hugging Face embeddings, PDF support, and Gradio.


In [ ]:
%pip install -qU \
  langchain \
  langchain-core \
  langchain-groq \
  langchain-text-splitters \
  langchain-huggingface \
  langchain-community \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  gradio

## Step 2 — Add Your Groq API Key

Recommended method in Colab:

1. Click **Secrets** from the left sidebar.
2. Add a new secret named:

```text
GROQ_API_KEY
```

3. Paste your Groq API key as the value.
4. Enable notebook access.

If the secret is not found, the notebook will ask you to enter the key securely.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    groq_key = userdata.get("Groq_API")
except Exception:
    groq_key = None

if not groq_key:
    groq_key = getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = groq_key
os.environ["GROQ_MODEL"] = "llama-3.1-8b-instant"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"

print("Groq key loaded:", "Yes" if os.environ.get("GROQ_API_KEY") else "No")
print("Groq model:", os.environ["GROQ_MODEL"])
print("Embedding model:", os.environ["EMBEDDING_MODEL"])

Groq key loaded: Yes
Groq model: llama-3.1-8b-instant
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3 — Import Libraries

This notebook uses:

- `ChatGroq` for Groq LLM response generation
- `FAISS` for vector search
- `HuggingFaceEmbeddings` for embeddings
- `Gradio` for the web UI
- `pypdf` for PDF reading


In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

DATA_DIR = Path("/content/data/raw")
VECTOR_DB_PATH = Path("/content/faiss_index")

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully.")
print("Data folder:", DATA_DIR)

Libraries imported successfully.
Data folder: /content/data/raw


## Step 4 — Create Sample Data

This creates a sample text document so students can run the RAG pipeline even without uploading a file.


In [ ]:
sample_text = """Generative AI Course Notes

Generative AI is a type of artificial intelligence that can create new content such as text, images, code, audio, summaries, and conversations.

LangChain is a framework used to build applications with Large Language Models. It helps connect prompts, models, tools, memory, documents, chains, and retrieval systems.

A RAG system means Retrieval-Augmented Generation. It retrieves relevant information from documents and then generates an answer using an LLM.

The main steps of a simple RAG pipeline are:
1. Data ingestion: load documents.
2. Data transformation: split documents into chunks.
3. Embeddings: convert text chunks into numerical vectors.
4. Vector database: store and search embeddings.
5. Retrieval: find relevant chunks for a user question.
6. LLM answer generation: use retrieved chunks to answer the question.

Groq provides fast inference for open-source language models. Gradio is used to create a simple web interface for AI applications.

Attention is a mechanism used in transformer models. It helps the model focus on the most relevant words or tokens when generating a response.
"""

sample_file = DATA_DIR / "sample_ai_notes.txt"
sample_file.write_text(sample_text, encoding="utf-8")

print("Sample file created:", sample_file)
print("Files in data folder:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Sample file created: /content/data/raw/sample_ai_notes.txt
Files in data folder:
- attention.pdf
- sample_ai_notes.txt


## Step 5 — Optional: Upload Your Own Files

You can upload PDF, TXT, MD, or XML files.

For classroom practice, students can upload their own notes, reports, or PDFs.


In [ ]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    file_path = DATA_DIR / filename
    file_path.write_bytes(content)
    print("Uploaded:", file_path)

print("\nCurrent files:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Saving attention.pdf to attention (1).pdf
Uploaded: /content/data/raw/attention (1).pdf

Current files:
- attention.pdf
- attention (1).pdf
- sample_ai_notes.txt


## Step 6 — Data Ingestion

This step loads documents from the `/content/data/raw` folder.

Supported file types:

- `.txt`
- `.md`
- `.pdf`
- `.xml`


In [ ]:
def read_txt_or_md(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf(path: Path):
    reader = PdfReader(str(path))
    docs = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "page": page_number,
                        "type": "pdf"
                    }
                )
            )

    return docs


def read_xml(path: Path) -> str:
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        text_parts = []

        for element in root.iter():
            if element.text and element.text.strip():
                text_parts.append(element.text.strip())

        return "\n".join(text_parts)
    except Exception:
        return path.read_text(encoding="utf-8", errors="ignore")


def load_documents_from_directory(directory=DATA_DIR):
    documents = []
    supported_extensions = {".txt", ".md", ".pdf", ".xml"}

    for path in sorted(Path(directory).rglob("*")):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix not in supported_extensions:
            continue

        if suffix == ".pdf":
            documents.extend(read_pdf(path))

        elif suffix in {".txt", ".md"}:
            text = read_txt_or_md(path)
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "type": suffix.replace(".", "")
                    }
                )
            )

        elif suffix == ".xml":
            text = read_xml(path)
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "type": "xml"
                    }
                )
            )

    return documents


documents = load_documents_from_directory(DATA_DIR)

print("STEP 1: DATA INGESTION")
print("=" * 50)
print("Documents loaded:", len(documents))

for i, doc in enumerate(documents[:3], start=1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print("Source:", doc.metadata.get("source"))
    print("Type:", doc.metadata.get("type"))
    print(doc.page_content[:500])

STEP 1: DATA INGESTION
Documents loaded: 16

Document 1
--------------------------------------------------
Source: attention.pdf
Type: pdf
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

Document 2
--------------------------------------------------
Source: attention.pdf
Type: pdf
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks
in particular, have been firmly established as state of the art approaches in sequence modeling and
transduction problems such as language modeling and mac

## Step 7 — Data Transformation

Now we split long documents into smaller chunks.

Why chunking is important:

- LLMs cannot always process very long files at once.
- Smaller chunks help retrieval.
- FAISS searches chunks, not the full document.


In [ ]:
def split_documents(documents, chunk_size=800, chunk_overlap=120):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    return splitter.split_documents(documents)


chunks = split_documents(documents, chunk_size=800, chunk_overlap=120)

print("STEP 2: DATA TRANSFORMATION")
print("=" * 50)
print("Original documents:", len(documents))
print("Chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}")
    print("-" * 50)
    print("Source:", chunk.metadata.get("source"))
    print(chunk.page_content[:500])

STEP 2: DATA TRANSFORMATION
Original documents: 16
Chunks created: 68

Chunk 1
--------------------------------------------------
Source: attention.pdf
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

Chunk 2
--------------------------------------------------
Source: attention.pdf
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms,

## Step 8 — Embeddings and FAISS Vector Database

This step converts text chunks into numerical vectors.

Then FAISS stores those vectors for fast similarity search.

The first run may take some time because the embedding model will download.


In [ ]:
EMBEDDING_MODEL = os.environ.get(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(str(VECTOR_DB_PATH))

print("STEP 3: EMBEDDINGS + FAISS")
print("=" * 50)
print("Embedding model:", EMBEDDING_MODEL)
print("Chunks embedded:", len(chunks))
print("Vector database saved at:", VECTOR_DB_PATH)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

STEP 3: EMBEDDINGS + FAISS
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Chunks embedded: 68
Vector database saved at: /content/faiss_index


## Step 9 — Test Vector Search

Before using Groq, test whether FAISS can retrieve relevant chunks.


In [ ]:
question = "what is the capital of Pakistan?"

results = vector_store.similarity_search(question, k=3)

print("STEP 4: VECTOR SEARCH")
print("=" * 50)
print("Question:", question)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 50)
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])

STEP 4: VECTOR SEARCH
Question: what is the capital of Pakistan?

Result 1
--------------------------------------------------
Source: attention.pdf
Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5
and 6. Note that the attentions are very sharp for this word.
14

Result 2
--------------------------------------------------
Source: attention.pdf
Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base
model. All metrics are on the English-to-German translation development set, newstest2013. Listed
perplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to
per-word perplexities.
N d model dff h d k dv Pdrop ϵls
train PPL BLEU params
steps (dev) (dev) ×106
base 6 512 2048 8 64 64 0.1 0.1 100K 4.92 25.8 65
(A)
1 512 512 5.29 24.9
4 128 128 5.00 25.5
16 32 32 4.91 25.8
32 16 16 5.01 25.4
(B) 16 5.16 25.1 58
32 5.01 25.4 60
(C)
2 6.11 23.7 36
4 5.19 25.3 

## Step 10 — Create Groq RAG Chain

Now we combine:

```text
User Question → FAISS Retrieval → Context → Groq LLM → Final Answer
```


In [ ]:
def get_llm(temperature=0.1):
    return ChatGroq(
        model=os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant"),
        temperature=temperature,
        api_key=os.environ.get("GROQ_API_KEY")
    )


def answer_with_groq(question, k=3, temperature=0.1):
    retrieved_docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}] {doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful PGD Generative AI teaching assistant. "
                "Answer using only the given context. "
                "Use simple classroom language. "
                "If the answer is not in the context, say: "
                "'I do not know from the uploaded documents.'"
            ),
            (
                "human",
                "Context:\n{context}\n\nQuestion:\n{question}\n\n"
                "Give a clear answer."
            )
        ]
    )

    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke(
        {
            "context": context,
            "question": question
        }
    )

    sources = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        label = f"{source}, page {page}" if page else source
        if label not in sources:
            sources.append(label)

    return answer, sources


answer, sources = answer_with_groq("What is LangChain?", k=3)

print("Groq Answer")
print("=" * 50)
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Groq Answer
LangChain is a framework used to build applications with Large Language Models. It helps connect prompts, models, tools, memory, documents, chains, and retrieval systems.

Sources:
- sample_ai_notes.txt
- attention.pdf, page 12


## Step 11 — Ask Your Own Question

Change the question below and run the cell again.


In [ ]:
my_question = "What are the steps of a RAG pipeline?"

answer, sources = answer_with_groq(my_question, k=4, temperature=0.1)

print("Question:", my_question)
print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Question: What are the steps of a RAG pipeline?

Answer:
The main steps of a simple RAG pipeline are:

1. Data ingestion: load documents.
2. Data transformation: split documents into chunks.
3. Embeddings: convert text chunks into numerical vectors.
4. Vector database: store and search embeddings.
5. Retrieval: find relevant chunks for a user question.
6. LLM answer generation: use retrieved chunks to answer the question.

Sources:
- sample_ai_notes.txt
- attention.pdf, page 9
- attention.pdf, page 8
- attention.pdf, page 13


## Step 12 — Groq Chat Playground

This is a simple Groq chat without RAG.

Use it to explain prompting and temperature.


In [ ]:
def simple_groq_chat(message, temperature=0.4):
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful Generative AI assistant. "
                "Explain concepts simply for students."
            ),
            ("human", "{message}")
        ]
    )

    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()

    return chain.invoke({"message": message})


print(simple_groq_chat("Explain LangChain in simple words.", temperature=0.4))

**What is LangChain?**

LangChain is an open-source library that helps developers build conversational AI systems. It's like a toolkit that makes it easier to create chatbots, virtual assistants, and other language-based applications.

**Key Features of LangChain:**

1. **Chainable Functions**: LangChain allows you to create a series of functions that work together to process user input and generate responses. These functions are "chained" together, making it easy to build complex conversational flows.
2. **Memory and Context**: LangChain provides a way to store and retrieve information from previous conversations, allowing your AI system to remember user preferences, history, and context.
3. **Natural Language Processing (NLP)**: LangChain includes built-in NLP capabilities, such as text analysis, entity recognition, and sentiment analysis, making it easier to understand user input and generate relevant responses.
4. **Integration with Other Libraries**: LangChain can be used with oth

## Step 13 — Appealing Gradio UI App

This app has two tabs:

1. **Ask Your Documents** — RAG-based question answering
2. **Groq Chat Playground** — normal Groq chatbot

In Colab, `share=True` creates a public temporary Gradio link.


In [ ]:
CUSTOM_CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}
#hero {
    padding: 26px;
    border-radius: 22px;
    background: linear-gradient(135deg, #0f172a, #1e293b, #172554);
    color: white;
    box-shadow: 0 18px 60px rgba(0,0,0,.22);
    margin-bottom: 18px;
}
#hero h1 {
    font-size: 38px;
    margin-bottom: 8px;
    background: linear-gradient(90deg, #22c55e, #38bdf8, #a78bfa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
#hero p {
    color: #cbd5e1;
    font-size: 16px;
}
"""


def rag_ui_answer(question, k, temperature):
    if not question or not question.strip():
        return "Please write a question."

    try:
        answer, sources = answer_with_groq(
            question=question,
            k=int(k),
            temperature=float(temperature)
        )

        source_text = "\n".join(f"- {source}" for source in sources)

        return f"""{answer}

### Sources
{source_text}
"""

    except Exception as e:
        return f"""Error: {str(e)}

Please check:
1. Groq API key is set
2. Documents are loaded
3. Vector database was created
"""


def playground_ui(message, temperature):
    if not message or not message.strip():
        return "Please write a prompt."

    try:
        return simple_groq_chat(message, temperature=float(temperature))
    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(css=CUSTOM_CSS, title="Groq RAG Studio", theme=gr.themes.Soft()) as demo:
    gr.HTML(
        """
        <div id="hero">
            <h1>⚡ Groq RAG Studio</h1>
            <p>Google Colab version of LangChain Framework using Groq API, FAISS, embeddings, and Gradio.</p>
            <p>Ask questions from your uploaded notes, PDFs, TXT, MD, or XML files.</p>
        </div>
        """
    )

    with gr.Tabs():
        with gr.Tab("📚 Ask Your Documents"):
            gr.Markdown("Ask questions from the documents loaded into FAISS.")

            question_box = gr.Textbox(
                label="Your Question",
                placeholder="Example: What is RAG? What is LangChain? What is attention?",
                lines=3
            )

            with gr.Row():
                k_slider = gr.Slider(1, 8, value=4, step=1, label="Retrieved Chunks")
                temp_slider = gr.Slider(0, 1, value=0.1, step=0.1, label="Temperature")

            ask_btn = gr.Button("Ask Groq", variant="primary")
            output_box = gr.Markdown(label="Answer")

            ask_btn.click(
                fn=rag_ui_answer,
                inputs=[question_box, k_slider, temp_slider],
                outputs=output_box
            )

        with gr.Tab("🧠 Groq Chat Playground"):
            gr.Markdown("Use this tab to test normal Groq prompting without documents.")

            prompt_box = gr.Textbox(
                label="Prompt",
                placeholder="Explain prompt engineering in simple words.",
                lines=5
            )

            playground_temp = gr.Slider(0, 1, value=0.4, step=0.1, label="Temperature")
            generate_btn = gr.Button("Generate with Groq", variant="primary")
            playground_output = gr.Textbox(label="Groq Response", lines=12)

            generate_btn.click(
                fn=playground_ui,
                inputs=[prompt_box, playground_temp],
                outputs=playground_output
            )

        with gr.Tab("🧩 RAG Pipeline Explanation"):
            gr.Markdown(
                """
                ## RAG Pipeline Used in This Notebook

                1. **Data Ingestion**
                   Load PDF, TXT, MD, and XML files.

                2. **Data Transformation**
                   Split long text into smaller chunks.

                3. **Embeddings**
                   Convert chunks into numerical vectors.

                4. **FAISS Vector Database**
                   Store and search similar chunks.

                5. **Retrieval**
                   Retrieve the most relevant chunks for the user's question.

                6. **Groq LLM**
                   Generate an answer using retrieved context.

                7. **Gradio UI**
                   Provide an easy web interface.
                """
            )

demo.launch(share=True)

/tmp/ipykernel_578/2812728530.py:67: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Groq RAG Studio", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb4f88193de452e73f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Student Task

Students should try the following:

1. Upload a PDF or TXT file.
2. Re-run the ingestion cell.
3. Re-run the transformation cell.
4. Re-run the embedding cell.
5. Ask questions from the uploaded document.
6. Compare answers at different temperature values.
7. Explain each RAG step in their own words.
